# 28 — Run LIANA condition-specific (memory-safe canonical run)

**Purpose:** reproduce the historical condition-specific LIANA analysis using the fixed canonical labels and cell identities produced by Notebook 25.

Historical counterpart:
- `25_run_LIANA_condition_specific_HVG8000_r0_6_v2_fixed_use_raw_false.ipynb`

### Canonicalization policy
- The biological membership and historical inclusion logic are preserved.
- Historical `NonMalignant_*` display labels are replaced by the canonical Notebook 25 labels.
- `B_Plasma` and `Mast_cell` remain excluded from the condition-specific main LIANA runs because that was the actual historical Notebook 25 condition-specific analysis behavior.
- The per-condition minimum support threshold remains **50 cells**.
- `magnitude_rank <= 0.05` remains the prioritization rule.
- No upstream notebook is rerun.

### Memory-safety design
- The full 351,427-cell AnnData is **not** loaded into RAM.
- Canonical cell IDs/labels/conditions are read from Notebook 25 `cell_metadata.tsv`.
- LIANA uses the canonical Notebook 25 labels directly. A controlled LGG diagnostic confirmed that substituting the clean historical `NonMalignant_*` aliases changes neither interactions nor scores; historical aliases are therefore retained only as an output-hygiene mapping reference.
- Only one condition is loaded from the H5AD `counts` layer at a time.
- Sparse counts are streamed from the HDF5 CSR layer while preserving the original counts dtype.
- The exact historical Scanpy preprocessing is reproduced with `sc.pp.normalize_total(target_sum=1e4) -> sc.pp.log1p` on the minimal per-condition object.
- A minimal AnnData containing only `X`, one LIANA grouping column, and `var_names` is created.
- Each condition expression object is deleted immediately after its LIANA result table is copied.
- LIANA execution is pinned to one job.

Run **top to bottom in a fresh kernel**.


In [1]:
# Environment
import sys, os, gc, platform, warnings
from pathlib import Path
from datetime import datetime

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Started:", datetime.now().isoformat(timespec="seconds"))


Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
Started: 2026-09-19T03:47:35


In [2]:
# Imports
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import h5py
import matplotlib.pyplot as plt

try:
    import psutil
    PSUTIL_AVAILABLE = True
except Exception:
    PSUTIL_AVAILABLE = False

try:
    import liana as li
    LIANA_AVAILABLE = True
    LIANA_VERSION = str(getattr(li, "__version__", "unknown_version"))
    SCANPY_VERSION = str(getattr(sc, "__version__", "unknown_version"))
    print("[OK] liana:", LIANA_VERSION)
    print("[OK] scanpy:", SCANPY_VERSION)
except Exception as e:
    LIANA_AVAILABLE = False
    LIANA_VERSION = "unavailable"
    SCANPY_VERSION = str(getattr(sc, "__version__", "unknown_version"))
    print("[ERROR] Could not import liana:", repr(e))


[OK] liana: 1.7.1
[OK] scanpy: 1.11.5


In [3]:
# Fixed paths and canonical/historical configuration
def _find_repository_root(start=None):
    # Resolve repository root only; analytical filenames remain fixed.
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()

METADATA_DIR = PROJECT_DIR / "metadata"
FIGURE_DIR = PROJECT_DIR / "figures" / "LIANA_condition_specific_HVG8000_r0_6"
OUT_DIR = PROJECT_DIR / "data" / "processed" / "integrated"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = OUT_DIR / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"
CELL_METADATA_TSV = METADATA_DIR / "LIANA_prepare_HVG8000_r0_6_cell_metadata.tsv"
NB25_CONDITION_COUNTS_TSV = METADATA_DIR / "LIANA_prepare_HVG8000_r0_6_condition_celltype_counts.tsv"

OUT_PREFIX = "LIANA_condition_specific_HVG8000_r0_6"

CONDITIONS_ORDER = ["GBM", "LGG", "ndGBM", "rGBM"]
COUNTS_LAYER = "counts"
GROUPBY = "liana_celltype_condition_main"

# Historical condition-specific membership semantics.
MIN_CELLS_PER_CELLTYPE_CONDITION = 50
HISTORICAL_EXCLUDED_CELLTYPES = {"B_Plasma", "Mast_cell"}

CANONICAL_MALIGNANT_TYPES = {
    "Malignant_OPC_NPC_like_malignant",
    "Malignant_cycling_candidate",
    "Malignant_hypoxia_stress_malignant",
    "Malignant_AC_like_reactive_malignant",
    "Malignant_NPC_neuronal_like_malignant",
}
CANONICAL_MICROENV_TYPES = {
    "Oligodendrocyte_myelinating",
    "Mural_Pericyte_Fibroblast",
    "Endothelial",
}
CANONICAL_CONDITION_MAIN_TYPES = CANONICAL_MALIGNANT_TYPES | CANONICAL_MICROENV_TYPES

# Historical Notebook 25 used these clean NonMalignant display labels.
# Notebook 28 computes directly with canonical Notebook 25 labels. This mapping is
# retained only to prevent legacy aliases from leaking into outputs if encountered.
CANONICAL_TO_HISTORICAL_COMPUTATION_LABEL = {
    "Endothelial": "NonMalignant_endothelial_reference_like",
    "Mural_Pericyte_Fibroblast": "NonMalignant_mural_pericyte_fibroblast_like",
    "Oligodendrocyte_myelinating": "NonMalignant_oligodendrocyte_myelinating_reference_like",
}
HISTORICAL_COMPUTATION_TO_CANONICAL_LABEL = {
    v: k for k, v in CANONICAL_TO_HISTORICAL_COMPUTATION_LABEL.items()
}
for _m in CANONICAL_MALIGNANT_TYPES:
    CANONICAL_TO_HISTORICAL_COMPUTATION_LABEL[_m] = _m
    HISTORICAL_COMPUTATION_TO_CANONICAL_LABEL[_m] = _m

# Preserve the historical categorical iteration order after canonical relabeling.
HISTORICAL_CATEGORY_ORDER_CANONICAL = [
    "Malignant_AC_like_reactive_malignant",
    "Malignant_NPC_neuronal_like_malignant",
    "Malignant_OPC_NPC_like_malignant",
    "Malignant_cycling_candidate",
    "Malignant_hypoxia_stress_malignant",
    "Endothelial",
    "Mural_Pericyte_Fibroblast",
    "Oligodendrocyte_myelinating",
]

# LIANA settings. These make the historical implicit defaults explicit.
RESOURCE_NAME = "consensus"
EXPR_PROP = 0.10
LIANA_MIN_CELLS = 5
N_PERMS = 1000
SEED = 1337
N_JOBS = 1
EXPECTED_LIANA_VERSION = "1.7.1"

# Expression preprocessing
NORMALIZATION_TARGET_SUM = 1e4
H5_ROW_CHUNK = 512

# Result selection
EXPECTED_PRIMARY_RANK = "magnitude_rank"
RANK_THRESHOLD = 0.05
TOP_N_GLOBAL_EXPORT_PER_CONDITION = 500

# Fixed canonical input checkpoints
EXPECTED_INPUT_N_OBS = 351_427
EXPECTED_INPUT_N_VARS = 15_176

# Historical execution checkpoints — audit-only; never used to manufacture results.
EXPECTED_RUN_PLAN = {
    "GBM":   {"n_cells": 93_563, "n_celltypes": 8, "n_liana_rows": 11_094, "n_prioritized": 662},
    "LGG":   {"n_cells": 6_333,  "n_celltypes": 6, "n_liana_rows": 11_050, "n_prioritized": 625},
    "ndGBM": {"n_cells": 49_830, "n_celltypes": 8, "n_liana_rows": 23_603, "n_prioritized": 1_337},
    "rGBM":  {"n_cells": 33_099, "n_celltypes": 8, "n_liana_rows": 30_423, "n_prioritized": 1_676},
}
EXPECTED_TOTAL_LIANA_ROWS = 76_170
EXPECTED_TOTAL_PRIORITIZED = 4_300
EXPECTED_PRESENCE_ROWS = 2_450
EXPECTED_FOCUSED_ROWS = 3_465
EXPECTED_CANDIDATE_ROWS = 2_302
EXPECTED_CANDIDATE_SUMMARY_ROWS = 475

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_H5AD:", INPUT_H5AD)
print("CELL_METADATA_TSV:", CELL_METADATA_TSV)


PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
INPUT_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_malignant_states.h5ad
CELL_METADATA_TSV: G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_cell_metadata.tsv


In [4]:
# Helpers
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print("[WRITE]", path, df.shape)
    return path

def savefig(path, dpi=180):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("[FIG]", path)
    return path

def safe_read_tsv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))
    return pd.read_csv(path, sep="\t", **kwargs)

def available_ram_gb():
    if not PSUTIL_AVAILABLE:
        return np.nan
    return psutil.virtual_memory().available / (1024 ** 3)

def _decode_attr(x):
    if isinstance(x, bytes):
        return x.decode()
    return str(x)

def read_h5ad_axis_index(h5ad_path, axis):
    # Read only obs/var index strings directly from HDF5.
    with h5py.File(h5ad_path, "r") as h5:
        if axis not in h5:
            raise KeyError(f"H5AD axis group {axis!r} is missing.")
        g = h5[axis]
        idx_key = _decode_attr(g.attrs.get("_index", "_index"))
        if idx_key not in g:
            raise KeyError(f"H5AD {axis!r} index dataset {idx_key!r} is missing.")
        values = g[idx_key][:]
    decoded = [
        x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x)
        for x in values
    ]
    return pd.Index(decoded)

def inspect_csr_layer(h5ad_path, layer_name, row_positions):
    row_positions = np.asarray(row_positions, dtype=np.int64)
    if row_positions.ndim != 1 or len(row_positions) == 0:
        raise ValueError("row_positions must be a non-empty 1D array.")
    if np.any(np.diff(row_positions) <= 0):
        raise ValueError("row_positions must be strictly increasing and unique.")

    with h5py.File(h5ad_path, "r") as h5:
        key = f"layers/{layer_name}"
        if key not in h5:
            raise KeyError(f"Required layer {layer_name!r} not found.")
        g = h5[key]
        enc = _decode_attr(g.attrs.get("encoding-type", ""))
        if enc != "csr_matrix":
            raise RuntimeError(
                f"Memory-safe loader requires CSR-encoded {layer_name!r}; "
                f"found encoding-type={enc!r}."
            )
        shape = tuple(int(x) for x in g.attrs["shape"])
        if row_positions[0] < 0 or row_positions[-1] >= shape[0]:
            raise IndexError("Selected row positions are outside layer bounds.")
        indptr = np.asarray(g["indptr"][:], dtype=np.int64)
        row_nnz = indptr[row_positions + 1] - indptr[row_positions]
        selected_nnz = int(row_nnz.sum())
        indices_dtype = np.dtype(g["indices"].dtype)
        source_data_dtype = np.dtype(g["data"].dtype)

    target_bytes = (
        selected_nnz * (source_data_dtype.itemsize + indices_dtype.itemsize)
        + (len(row_positions) + 1) * np.dtype(np.int64).itemsize
    )
    return {
        "shape": shape,
        "encoding_type": enc,
        "selected_nnz": selected_nnz,
        "source_data_dtype": str(source_data_dtype),
        "indices_dtype": str(indices_dtype),
        "target_csr_gb": target_bytes / (1024 ** 3),
    }

def load_selected_counts_csr(h5ad_path, layer_name, row_positions, chunk_rows=512):
    # Stream selected CSR rows while preserving the source counts dtype.
    row_positions = np.asarray(row_positions, dtype=np.int64)
    meta = inspect_csr_layer(h5ad_path, layer_name, row_positions)
    n_rows_src, n_vars = meta["shape"]

    selected_mask = np.zeros(n_rows_src, dtype=bool)
    selected_mask[row_positions] = True

    with h5py.File(h5ad_path, "r") as h5:
        g = h5[f"layers/{layer_name}"]
        indptr_src = np.asarray(g["indptr"][:], dtype=np.int64)

        row_nnz = indptr_src[row_positions + 1] - indptr_src[row_positions]
        out_indptr = np.empty(len(row_positions) + 1, dtype=np.int64)
        out_indptr[0] = 0
        np.cumsum(row_nnz, out=out_indptr[1:])
        total_nnz = int(out_indptr[-1])

        out_data = np.empty(total_nnz, dtype=g["data"].dtype)
        out_indices = np.empty(total_nnz, dtype=g["indices"].dtype)

        cursor = 0
        for start in range(0, n_rows_src, chunk_rows):
            stop = min(start + chunk_rows, n_rows_src)
            rel = np.flatnonzero(selected_mask[start:stop])
            if len(rel) == 0:
                continue

            p0 = int(indptr_src[start])
            p1 = int(indptr_src[stop])

            chunk_data = np.asarray(g["data"][p0:p1])
            chunk_indices = np.asarray(g["indices"][p0:p1])
            chunk_indptr = indptr_src[start:stop + 1] - p0

            chunk = sp.csr_matrix(
                (chunk_data, chunk_indices, chunk_indptr),
                shape=(stop - start, n_vars),
            )
            sub = chunk[rel]

            n = int(sub.nnz)
            out_data[cursor:cursor + n] = sub.data
            out_indices[cursor:cursor + n] = sub.indices
            cursor += n

            del chunk_data, chunk_indices, chunk_indptr, chunk, sub

        if cursor != total_nnz:
            raise RuntimeError(f"CSR extraction mismatch: copied {cursor} != expected {total_nnz}")

    X = sp.csr_matrix(
        (out_data, out_indices, out_indptr),
        shape=(len(row_positions), n_vars),
        copy=False,
    )
    if not X.has_sorted_indices:
        X.sort_indices()

    del selected_mask
    gc.collect()
    return X, meta

def lineage_simplified(celltype):
    c = str(celltype)
    if c in CANONICAL_MALIGNANT_TYPES:
        return "malignant_or_cycling"
    if c in CANONICAL_MICROENV_TYPES:
        return "microenvironment"
    return "excluded_or_review"

BIOLOGICAL_GROUP_MAP = {
    "Malignant_OPC_NPC_like_malignant": "malignant_OPC_NPC",
    "Malignant_cycling_candidate": "malignant_cycling",
    "Malignant_hypoxia_stress_malignant": "malignant_hypoxia_stress",
    "Malignant_AC_like_reactive_malignant": "malignant_AC_reactive",
    "Malignant_NPC_neuronal_like_malignant": "malignant_NPC_neuronal",
    "Endothelial": "endothelial",
    "Mural_Pericyte_Fibroblast": "mural_pericyte_fibroblast",
    "Oligodendrocyte_myelinating": "oligodendrocyte_reference",
}

def biological_group(celltype):
    return BIOLOGICAL_GROUP_MAP.get(str(celltype), "excluded_or_review")

def axis_from_source_target(source, target):
    s = lineage_simplified(source)
    t = lineage_simplified(target)
    if s == "malignant_or_cycling" and t == "malignant_or_cycling":
        return "malignant_or_cycling_to_malignant_or_cycling"
    if s == "malignant_or_cycling" and t == "microenvironment":
        return "malignant_or_cycling_to_microenvironment"
    if s == "microenvironment" and t == "malignant_or_cycling":
        return "microenvironment_to_malignant_or_cycling"
    if s == "microenvironment" and t == "microenvironment":
        return "microenvironment_to_microenvironment"
    return "other_or_excluded"

def biological_axis_from_source_target(source, target):
    return f"{biological_group(source)}_to_{biological_group(target)}"

def add_liana_summary_columns(df):
    required = {"source", "target", "ligand_complex", "receptor_complex", EXPECTED_PRIMARY_RANK}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"LIANA result schema drift. Missing: {sorted(missing)}")
    df = df.copy()

    # Defensive output hygiene: canonicalize any legacy aliases if encountered.
    df["source"] = (
        df["source"].astype(str)
        .map(lambda x: HISTORICAL_COMPUTATION_TO_CANONICAL_LABEL.get(x, x))
    )
    df["target"] = (
        df["target"].astype(str)
        .map(lambda x: HISTORICAL_COMPUTATION_TO_CANONICAL_LABEL.get(x, x))
    )

    df["ligand"] = df["ligand_complex"].astype(str)
    df["receptor"] = df["receptor_complex"].astype(str)
    df["lr_pair"] = df["ligand"] + "→" + df["receptor"]
    df["source_clean"] = df["source"].astype(str)
    df["target_clean"] = df["target"].astype(str)
    df["source_lineage"] = df["source_clean"].map(lineage_simplified)
    df["target_lineage"] = df["target_clean"].map(lineage_simplified)
    df["axis"] = [
        axis_from_source_target(s, t)
        for s, t in zip(df["source_clean"], df["target_clean"])
    ]
    df["biological_axis"] = [
        biological_axis_from_source_target(s, t)
        for s, t in zip(df["source_clean"], df["target_clean"])
    ]
    df["source_target"] = df["source_clean"] + " → " + df["target_clean"]
    df["interaction_key"] = (
        df["source_clean"] + "|" + df["target_clean"] + "|" +
        df["ligand"] + "|" + df["receptor"]
    )
    df["rank_for_sorting"] = pd.to_numeric(df[EXPECTED_PRIMARY_RANK], errors="coerce")
    if df["rank_for_sorting"].isna().any():
        raise ValueError("NA/non-numeric magnitude_rank values detected.")
    return df

def summarize_source_target(df):
    if len(df) == 0:
        return pd.DataFrame()
    return (
        df.groupby(
            ["condition", "source_clean", "target_clean", "source_target", "axis", "biological_axis"],
            observed=True,
        )
        .agg(
            n_interactions=("lr_pair", "size"),
            n_unique_lr=("lr_pair", "nunique"),
            min_rank=("rank_for_sorting", "min"),
            median_rank=("rank_for_sorting", "median"),
            top_lr=("lr_pair", lambda x: ";".join(pd.Series(x).astype(str).head(5))),
        )
        .reset_index()
        .sort_values(["condition", "n_interactions", "min_rank"], ascending=[True, False, True])
    )

def summarize_axis(df, col="axis"):
    if len(df) == 0:
        return pd.DataFrame()
    return (
        df.groupby(["condition", col], observed=True)
        .agg(
            n_interactions=("lr_pair", "size"),
            n_unique_source_target=("source_target", "nunique"),
            n_unique_lr=("lr_pair", "nunique"),
            min_rank=("rank_for_sorting", "min"),
            median_rank=("rank_for_sorting", "median"),
        )
        .reset_index()
        .sort_values(["condition", "n_interactions"], ascending=[True, False])
    )

def summarize_lr(df):
    if len(df) == 0:
        return pd.DataFrame()
    return (
        df.groupby(["condition", "lr_pair", "ligand", "receptor"], observed=True)
        .agg(
            n_source_target_pairs=("source_target", "nunique"),
            n_interaction_rows=("lr_pair", "size"),
            min_rank=("rank_for_sorting", "min"),
            median_rank=("rank_for_sorting", "median"),
            top_source_target=("source_target", lambda x: ";".join(pd.Series(x).astype(str).head(5))),
        )
        .reset_index()
        .sort_values(["condition", "n_source_target_pairs", "min_rank"], ascending=[True, False, True])
    )


In [5]:
# Readiness and exact canonical input schema
required_paths = [INPUT_H5AD, CELL_METADATA_TSV, NB25_CONDITION_COUNTS_TSV]
readiness = pd.DataFrame([
    {
        "item": p.name,
        "path": str(p),
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / 1024 / 1024, 3) if p.exists() else np.nan,
    }
    for p in required_paths
] + [
    {
        "item": "liana_version",
        "path": LIANA_VERSION,
        "exists": LIANA_AVAILABLE and LIANA_VERSION == EXPECTED_LIANA_VERSION,
        "size_mb": np.nan,
    }
])
write_tsv_replace(readiness, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")
display(readiness)

if not LIANA_AVAILABLE:
    raise ImportError("LIANA is not available in this environment.")
if LIANA_VERSION != EXPECTED_LIANA_VERSION:
    raise RuntimeError(
        f"Expected LIANA {EXPECTED_LIANA_VERSION} for the canonical LIANA phase; "
        f"found {LIANA_VERSION}."
    )
for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(str(p))

cell_meta = safe_read_tsv(
    CELL_METADATA_TSV,
    usecols=["cell_id", "condition", "liana_celltype_main"],
)

required_cols = {"cell_id", "condition", "liana_celltype_main"}
if set(cell_meta.columns) != required_cols:
    raise ValueError(f"Unexpected canonical cell metadata schema: {list(cell_meta.columns)}")
if cell_meta["cell_id"].isna().any() or cell_meta["cell_id"].duplicated().any():
    raise ValueError("Canonical cell_metadata has NA/duplicate cell_id values.")
if cell_meta["condition"].isna().any():
    raise ValueError("Canonical cell_metadata has NA condition values.")

observed_conditions = set(cell_meta["condition"].astype(str).unique())
if observed_conditions != set(CONDITIONS_ORDER):
    raise AssertionError(
        f"Condition set mismatch. Expected {CONDITIONS_ORDER}; observed {sorted(observed_conditions)}"
    )

resolved = pd.DataFrame([
    {"field": "cell_id", "column": "cell_id", "source": CELL_METADATA_TSV.name},
    {"field": "condition", "column": "condition", "source": CELL_METADATA_TSV.name},
    {"field": "LIANA cell type", "column": "liana_celltype_main", "source": CELL_METADATA_TSV.name},
    {"field": "expression", "column": COUNTS_LAYER, "source": INPUT_H5AD.name},
])
write_tsv_replace(resolved, METADATA_DIR / f"{OUT_PREFIX}_resolved_columns.tsv")
display(resolved)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_readiness.tsv (4, 4)


,item,path,exists,size_mb
0,GBM_core_4datasets_final_annotation_malignant_...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,14469.275
1,LIANA_prepare_HVG8000_r0_6_cell_metadata.tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,85.034
2,LIANA_prepare_HVG8000_r0_6_condition_celltype_...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.006
3,liana_version,1.7.1,True,NaN


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_resolved_columns.tsv (4, 3)


,field,column,source
0,cell_id,cell_id,LIANA_prepare_HVG8000_r0_6_cell_metadata.tsv
1,condition,condition,LIANA_prepare_HVG8000_r0_6_cell_metadata.tsv
2,LIANA cell type,liana_celltype_main,LIANA_prepare_HVG8000_r0_6_cell_metadata.tsv
3,expression,counts,GBM_core_4datasets_final_annotation_malignant_...


In [6]:
# Canonical labels + historical condition-specific inclusion logic
candidate = cell_meta.copy()
candidate["liana_celltype_raw"] = candidate["liana_celltype_main"].fillna("Review_or_ambiguous_excluded").astype(str)

celltype_counts_all = (
    candidate["liana_celltype_raw"]
    .value_counts(dropna=False)
    .rename_axis("liana_celltype_raw")
    .reset_index(name="n_cells")
)
celltype_counts_all["include_in_condition_specific_main"] = (
    celltype_counts_all["liana_celltype_raw"].isin(CANONICAL_CONDITION_MAIN_TYPES)
)
celltype_counts_all["lineage_simplified"] = celltype_counts_all["liana_celltype_raw"].map(lineage_simplified)
celltype_counts_all["biological_group"] = celltype_counts_all["liana_celltype_raw"].map(biological_group)
celltype_counts_all["historical_exclusion_reason"] = np.where(
    celltype_counts_all["liana_celltype_raw"].eq("B_Plasma"),
    "historically excluded from condition-specific main analysis",
    np.where(
        celltype_counts_all["liana_celltype_raw"].eq("Mast_cell"),
        "historically excluded from condition-specific main analysis",
        np.where(
            celltype_counts_all["liana_celltype_raw"].eq("Review_or_ambiguous_excluded"),
            "review/ambiguous population excluded",
            "",
        ),
    ),
)
write_tsv_replace(celltype_counts_all, METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_all.tsv")

condition_celltype_counts = (
    candidate.groupby(["condition", "liana_celltype_raw"], observed=True)
    .size()
    .reset_index(name="n_cells")
)
condition_celltype_counts["excluded_by_label"] = (
    ~condition_celltype_counts["liana_celltype_raw"].isin(CANONICAL_CONDITION_MAIN_TYPES)
)
condition_celltype_counts["passes_min_cells"] = (
    condition_celltype_counts["n_cells"] >= MIN_CELLS_PER_CELLTYPE_CONDITION
)
condition_celltype_counts["include_condition_specific"] = (
    (~condition_celltype_counts["excluded_by_label"])
    & condition_celltype_counts["passes_min_cells"]
)
condition_celltype_counts["lineage_simplified"] = (
    condition_celltype_counts["liana_celltype_raw"].map(lineage_simplified)
)
condition_celltype_counts["biological_group"] = (
    condition_celltype_counts["liana_celltype_raw"].map(biological_group)
)
condition_celltype_counts = condition_celltype_counts.sort_values(
    ["condition", "include_condition_specific", "n_cells"],
    ascending=[True, False, False],
)
write_tsv_replace(
    condition_celltype_counts,
    METADATA_DIR / f"{OUT_PREFIX}_condition_celltype_inclusion_decisions.tsv",
)

condition_run_plan = (
    condition_celltype_counts.loc[condition_celltype_counts["include_condition_specific"]]
    .groupby("condition", observed=True)
    .agg(
        n_celltypes=("liana_celltype_raw", "nunique"),
        n_cells_included=("n_cells", "sum"),
        included_celltypes=(
            "liana_celltype_raw",
            lambda x: ";".join(
                [c for c in HISTORICAL_CATEGORY_ORDER_CANONICAL if c in set(map(str, x))]
            ),
        ),
    )
    .reset_index()
)

run_plan_audit_rows = []
for cond in CONDITIONS_ORDER:
    row = condition_run_plan.loc[condition_run_plan["condition"].astype(str).eq(cond)]
    if len(row) != 1:
        raise AssertionError(f"Missing/duplicate run-plan row for {cond}.")
    observed_cells = int(row.iloc[0]["n_cells_included"])
    observed_types = int(row.iloc[0]["n_celltypes"])
    exp = EXPECTED_RUN_PLAN[cond]
    run_plan_audit_rows.extend([
        {
            "condition": cond, "checkpoint": "n_cells",
            "expected_historical": exp["n_cells"], "observed": observed_cells,
            "match": observed_cells == exp["n_cells"],
        },
        {
            "condition": cond, "checkpoint": "n_celltypes",
            "expected_historical": exp["n_celltypes"], "observed": observed_types,
            "match": observed_types == exp["n_celltypes"],
        },
    ])

run_plan_audit = pd.DataFrame(run_plan_audit_rows)
if not run_plan_audit["match"].all():
    raise AssertionError("Canonical condition-specific membership does not reproduce the historical run plan.")

write_tsv_replace(condition_run_plan, METADATA_DIR / f"{OUT_PREFIX}_condition_run_plan.tsv")
write_tsv_replace(run_plan_audit, METADATA_DIR / f"{OUT_PREFIX}_historical_membership_checkpoint_audit.tsv")

label_audit = pd.DataFrame([
    {
        "historical_label": "NonMalignant_oligodendrocyte_myelinating_reference_like",
        "canonical_label": "Oligodendrocyte_myelinating",
        "condition_specific_status": "included when n_cells>=50",
    },
    {
        "historical_label": "NonMalignant_mural_pericyte_fibroblast_like",
        "canonical_label": "Mural_Pericyte_Fibroblast",
        "condition_specific_status": "included when n_cells>=50",
    },
    {
        "historical_label": "NonMalignant_endothelial_reference_like",
        "canonical_label": "Endothelial",
        "condition_specific_status": "included when n_cells>=50",
    },
    {
        "historical_label": "NonMalignant_B_plasma_like",
        "canonical_label": "B_Plasma",
        "condition_specific_status": "excluded to preserve historical analysis membership",
    },
    {
        "historical_label": "NonMalignant_rare_mast_cell_like_low_count",
        "canonical_label": "Mast_cell",
        "condition_specific_status": "excluded to preserve historical analysis membership",
    },
])
write_tsv_replace(label_audit, METADATA_DIR / f"{OUT_PREFIX}_historical_to_canonical_label_audit.tsv")

display(condition_run_plan)
display(run_plan_audit)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_celltype_counts_all.tsv (11, 6)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_condition_celltype_inclusion_decisions.tsv (44, 8)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_condition_run_plan.tsv (4, 4)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_historical_membership_checkpoint_audit.tsv (8, 5)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_historical_to_canonical_label_audit.tsv (5, 3)


,condition,n_celltypes,n_cells_included,included_celltypes
0,GBM,8,93563,Malignant_AC_like_reactive_malignant;Malignant...
1,LGG,6,6333,Malignant_AC_like_reactive_malignant;Malignant...
2,ndGBM,8,49830,Malignant_AC_like_reactive_malignant;Malignant...
3,rGBM,8,33099,Malignant_AC_like_reactive_malignant;Malignant...


,condition,checkpoint,expected_historical,observed,match
0,GBM,n_cells,93563,93563,True
1,GBM,n_celltypes,8,8,True
2,LGG,n_cells,6333,6333,True
3,LGG,n_celltypes,6,6,True
4,ndGBM,n_cells,49830,49830,True
5,ndGBM,n_celltypes,8,8,True
6,rGBM,n_cells,33099,33099,True
7,rGBM,n_celltypes,8,8,True


In [7]:
# H5AD structural validation without loading the full AnnData
obs_names = read_h5ad_axis_index(INPUT_H5AD, "obs")
var_names = read_h5ad_axis_index(INPUT_H5AD, "var")

if len(obs_names) != EXPECTED_INPUT_N_OBS or len(var_names) != EXPECTED_INPUT_N_VARS:
    raise AssertionError(
        f"Canonical H5AD shape mismatch: observed {len(obs_names):,} x {len(var_names):,}; "
        f"expected {EXPECTED_INPUT_N_OBS:,} x {EXPECTED_INPUT_N_VARS:,}."
    )
if not obs_names.is_unique:
    raise AssertionError("Canonical H5AD obs_names are not unique.")
if not var_names.is_unique:
    raise AssertionError("Canonical H5AD var_names are not unique.")

with h5py.File(INPUT_H5AD, "r") as h5:
    if f"layers/{COUNTS_LAYER}" not in h5:
        raise KeyError(f"Required counts layer {COUNTS_LAYER!r} is missing.")

all_canonical_ids = pd.Index(cell_meta["cell_id"].astype(str))
positions_all = obs_names.get_indexer(all_canonical_ids)
n_missing_all = int((positions_all < 0).sum())

input_membership_audit = pd.DataFrame([
    {"checkpoint": "input_n_obs", "expected": EXPECTED_INPUT_N_OBS, "observed": len(obs_names), "match": len(obs_names) == EXPECTED_INPUT_N_OBS},
    {"checkpoint": "input_n_vars", "expected": EXPECTED_INPUT_N_VARS, "observed": len(var_names), "match": len(var_names) == EXPECTED_INPUT_N_VARS},
    {"checkpoint": "canonical_cell_metadata_rows", "expected": EXPECTED_INPUT_N_OBS, "observed": len(cell_meta), "match": len(cell_meta) == EXPECTED_INPUT_N_OBS},
    {"checkpoint": "canonical_cell_ids_missing_from_h5ad", "expected": 0, "observed": n_missing_all, "match": n_missing_all == 0},
])
write_tsv_replace(input_membership_audit, METADATA_DIR / f"{OUT_PREFIX}_input_membership_audit.tsv")
display(input_membership_audit)

if not input_membership_audit["match"].all():
    raise AssertionError("Canonical cell IDs and H5AD obs_names are not in one-to-one agreement.")

# Release the full mapping vector; per-condition mappings will be done independently.
del positions_all, all_canonical_ids
gc.collect()


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_input_membership_audit.tsv (4, 4)


,checkpoint,expected,observed,match
0,input_n_obs,351427,351427,True
1,input_n_vars,15176,15176,True
2,canonical_cell_metadata_rows,351427,351427,True
3,canonical_cell_ids_missing_from_h5ad,0,0,True


40

In [8]:
# Explicit LIANA configuration provenance
run_config = pd.DataFrame([
    {"parameter": "resource_name", "value": RESOURCE_NAME},
    {"parameter": "expr_prop", "value": EXPR_PROP},
    {"parameter": "liana_min_cells", "value": LIANA_MIN_CELLS},
    {"parameter": "condition_min_cells_pre_filter", "value": MIN_CELLS_PER_CELLTYPE_CONDITION},
    {"parameter": "n_perms", "value": N_PERMS},
    {"parameter": "seed", "value": SEED},
    {"parameter": "n_jobs", "value": N_JOBS},
    {"parameter": "python_version", "value": __import__("sys").version.replace("\n", " ")},
    {"parameter": "numpy_version", "value": np.__version__},
    {"parameter": "pandas_version", "value": pd.__version__},
    {"parameter": "scipy_version", "value": scipy.__version__},
    {"parameter": "anndata_version", "value": str(getattr(ad, "__version__", "unknown_version"))},
    {"parameter": "scanpy_version", "value": SCANPY_VERSION},
    {"parameter": "liana_version", "value": LIANA_VERSION},
    {"parameter": "use_raw", "value": False},
    {"parameter": "layer_passed_to_liana", "value": "None"},
    {"parameter": "counts_layer_source", "value": COUNTS_LAYER},
    {"parameter": "normalization_target_sum", "value": NORMALIZATION_TARGET_SUM},
    {"parameter": "normalization_transform", "value": "log1p"},
    {"parameter": "groupby", "value": GROUPBY},
    {"parameter": "group_label_mode", "value": "canonical_notebook25_direct"},
    {"parameter": "historical_aliases_used_for_computation", "value": False},
    {"parameter": "primary_rank", "value": EXPECTED_PRIMARY_RANK},
    {"parameter": "rank_threshold", "value": RANK_THRESHOLD},
    {"parameter": "historically_excluded_celltypes", "value": ";".join(sorted(HISTORICAL_EXCLUDED_CELLTYPES))},
])
write_tsv_replace(run_config, METADATA_DIR / f"{OUT_PREFIX}_run_config.tsv")
display(run_config)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_run_config.tsv (25, 2)


,parameter,value
0,resource_name,consensus
1,expr_prop,0.1
2,liana_min_cells,5
3,condition_min_cells_pre_filter,50
4,n_perms,1000
5,seed,1337
6,n_jobs,1
7,python_version,"3.12.3 | packaged by conda-forge | (main, Apr ..."
8,numpy_version,2.3.5
9,pandas_version,2.3.3


In [9]:
# Run memory-safe LIANA separately for each condition
all_results = []
top_results = []
prioritized_results = []
run_records = []
memory_records = []
expression_records = []
historical_result_records = []

for cond in CONDITIONS_ORDER:
    print("\n" + "=" * 100)
    print("[CONDITION]", cond)

    cond_decisions = condition_celltype_counts.loc[
        condition_celltype_counts["condition"].astype(str).eq(cond)
        & condition_celltype_counts["include_condition_specific"]
    ].copy()

    included_celltypes = [
        c for c in HISTORICAL_CATEGORY_ORDER_CANONICAL
        if c in set(cond_decisions["liana_celltype_raw"].astype(str))
    ]
    exp = EXPECTED_RUN_PLAN[cond]

    if len(included_celltypes) != exp["n_celltypes"]:
        raise AssertionError(
            f"{cond}: included cell-type count drift: {len(included_celltypes)} != {exp['n_celltypes']}"
        )

    cond_meta = cell_meta.loc[
        cell_meta["condition"].astype(str).eq(cond)
        & cell_meta["liana_celltype_main"].astype(str).isin(included_celltypes),
        ["cell_id", "liana_celltype_main"],
    ].copy()

    if len(cond_meta) != exp["n_cells"]:
        raise AssertionError(f"{cond}: cell membership drift {len(cond_meta):,} != {exp['n_cells']:,}.")
    if cond_meta["cell_id"].duplicated().any() or cond_meta["cell_id"].isna().any():
        raise ValueError(f"{cond}: duplicate/NA cell IDs in canonical condition membership.")

    requested_ids = pd.Index(cond_meta["cell_id"].astype(str))
    positions = obs_names.get_indexer(requested_ids)
    missing_ids = requested_ids[positions < 0]
    if len(missing_ids):
        raise AssertionError(f"{cond}: {len(missing_ids)} selected cells are missing from the H5AD.")
    if len(np.unique(positions)) != len(positions):
        raise AssertionError(f"{cond}: H5AD row mapping is not one-to-one.")

    row_positions = np.sort(positions.astype(np.int64))
    selected_ids = obs_names[row_positions]
    label_map = cond_meta.set_index("cell_id")["liana_celltype_main"]
    selected_labels = label_map.reindex(selected_ids)
    if selected_labels.isna().any():
        raise AssertionError(f"{cond}: NA labels after exact H5AD-order reindexing.")

    category_order = [c for c in HISTORICAL_CATEGORY_ORDER_CANONICAL if c in included_celltypes]

    used_counts = (
        selected_labels.astype(str)
        .value_counts()
        .rename_axis("liana_celltype")
        .reset_index(name="n_cells")
    )
    used_counts.insert(0, "condition", cond)
    used_counts["historical_category_order"] = used_counts["liana_celltype"].map(
        {c: i for i, c in enumerate(category_order)}
    )
    used_counts = used_counts.sort_values("historical_category_order")
    write_tsv_replace(
        used_counts,
        METADATA_DIR / f"{OUT_PREFIX}_{cond}_celltype_counts_used.tsv",
    )

    # Memory preflight for this condition before expression values are loaded.
    layer_meta = inspect_csr_layer(INPUT_H5AD, COUNTS_LAYER, row_positions)
    if tuple(layer_meta["shape"]) != (EXPECTED_INPUT_N_OBS, EXPECTED_INPUT_N_VARS):
        raise AssertionError(f"{cond}: counts-layer shape drift: {layer_meta['shape']}")

    estimated_gb = float(layer_meta["target_csr_gb"])
    available_gb = available_ram_gb()
    # Conservative because rank_aggregate also allocates method-level working objects.
    recommended_gb = max(5.0, 2.2 * estimated_gb + 2.0)

    memory_records.append({
        "condition": cond,
        "n_cells": len(row_positions),
        "selected_counts_nnz": int(layer_meta["selected_nnz"]),
        "source_counts_dtype": layer_meta["source_data_dtype"],
        "target_expression_dtype": "float32",
        "estimated_expression_csr_gb": round(estimated_gb, 3),
        "available_ram_gb_before_load": round(available_gb, 3) if pd.notna(available_gb) else np.nan,
        "recommended_available_ram_gb": round(recommended_gb, 3),
        "n_jobs": N_JOBS,
    })

    if pd.notna(available_gb) and available_gb < recommended_gb:
        write_tsv_replace(
            pd.DataFrame(memory_records),
            METADATA_DIR / f"{OUT_PREFIX}_memory_preflight.tsv",
        )
        raise MemoryError(
            f"{cond}: only {available_gb:.2f} GB RAM available; "
            f"recommended >= {recommended_gb:.2f} GB. "
            "Close other applications and rerun from a fresh kernel."
        )

    counts_csr, _ = load_selected_counts_csr(
        INPUT_H5AD,
        COUNTS_LAYER,
        row_positions,
        chunk_rows=H5_ROW_CHUNK,
    )

    if counts_csr.shape != (exp["n_cells"], EXPECTED_INPUT_N_VARS):
        raise AssertionError(f"{cond}: loaded expression shape mismatch {counts_csr.shape}.")
    nnz_before = int(counts_csr.nnz)
    if nnz_before != int(layer_meta["selected_nnz"]):
        raise AssertionError(f"{cond}: CSR nnz changed during streaming.")

    obs_min = pd.DataFrame(
        {
            GROUPBY: pd.Categorical(
                selected_labels.astype(str).values,
                categories=category_order,
                ordered=False,
            )
        },
        index=selected_ids,
    )
    adata_cond = ad.AnnData(
        X=counts_csr,
        obs=obs_min,
        var=pd.DataFrame(index=var_names),
    )
    adata_cond.raw = None

    del counts_csr, obs_min
    gc.collect()

    # Match historical Scanpy preprocessing on this condition only.
    # Release the counts alias first so Scanpy can replace integer X without
    # retaining the original CSR through another reference.
    sc.pp.normalize_total(adata_cond, target_sum=NORMALIZATION_TARGET_SUM)
    sc.pp.log1p(adata_cond)

    nnz_after = int(adata_cond.X.nnz)
    if nnz_after != nnz_before:
        raise AssertionError(f"{cond}: normalize/log1p unexpectedly changed sparse support.")
    if np.any(~np.isfinite(adata_cond.X.data)):
        raise ValueError(f"{cond}: non-finite normalized expression values.")

    expression_records.append({
        "condition": cond,
        "n_cells": adata_cond.n_obs,
        "n_genes": adata_cond.n_vars,
        "n_celltypes": adata_cond.obs[GROUPBY].nunique(),
        "normalized_X_nnz": nnz_after,
        "X_dtype": str(adata_cond.X.dtype),
        "normalization_target_sum": NORMALIZATION_TARGET_SUM,
        "transform": "log1p",
    })

    print("[LIANA minimal object]", adata_cond)

    li.mt.rank_aggregate(
        adata_cond,
        groupby=GROUPBY,
        resource_name=RESOURCE_NAME,
        expr_prop=EXPR_PROP,
        min_cells=LIANA_MIN_CELLS,
        use_raw=False,
        layer=None,
        n_perms=N_PERMS,
        seed=SEED,
        n_jobs=N_JOBS,
        verbose=True,
        inplace=True,
        key_added="liana_res",
    )

    if "liana_res" not in adata_cond.uns:
        raise RuntimeError(f"{cond}: LIANA completed without adata.uns['liana_res'].")

    res = add_liana_summary_columns(adata_cond.uns["liana_res"].copy())
    res.insert(0, "condition", cond)

    observed_result_types = set(res["source_clean"]).union(set(res["target_clean"]))
    if observed_result_types != set(included_celltypes):
        raise AssertionError(
            f"{cond}: LIANA source/target labels differ from included canonical groups. "
            f"Missing={sorted(set(included_celltypes)-observed_result_types)} "
            f"Unexpected={sorted(observed_result_types-set(included_celltypes))}"
        )
    if (res["axis"] == "other_or_excluded").any():
        raise AssertionError(f"{cond}: unexpected excluded/other communication axis in LIANA results.")
    if any(str(x).startswith("NonMalignant_") for x in observed_result_types):
        raise AssertionError(f"{cond}: historical NonMalignant_* labels leaked into canonical output.")
    if any("Malignant_non_malignant" in str(x) for x in observed_result_types):
        raise AssertionError(f"{cond}: malformed historical labels leaked into canonical output.")

    res_sorted = res.sort_values("rank_for_sorting", ascending=True, kind="mergesort").copy()
    prioritized = res_sorted.loc[
        res_sorted["rank_for_sorting"] <= RANK_THRESHOLD
    ].copy()
    prioritized["prioritization_rule"] = f"{EXPECTED_PRIMARY_RANK}<={RANK_THRESHOLD}"
    top = res_sorted.head(TOP_N_GLOBAL_EXPORT_PER_CONDITION).copy()

    n_rows = len(res)
    n_prioritized = len(prioritized)

    historical_result_records.extend([
        {
            "condition": cond, "checkpoint": "n_liana_rows",
            "expected_historical": exp["n_liana_rows"], "observed": n_rows,
            "match": n_rows == exp["n_liana_rows"],
        },
        {
            "condition": cond, "checkpoint": "n_prioritized",
            "expected_historical": exp["n_prioritized"], "observed": n_prioritized,
            "match": n_prioritized == exp["n_prioritized"],
        },
    ])

    all_results.append(res)
    top_results.append(top)
    prioritized_results.append(prioritized)

    run_records.append({
        "condition": cond,
        "status": "completed",
        "reason": "",
        "n_cells": adata_cond.n_obs,
        "n_genes": adata_cond.n_vars,
        "n_celltypes": adata_cond.obs[GROUPBY].nunique(),
        "included_celltypes": ";".join(category_order),
        "n_liana_rows": n_rows,
        "n_top_exported": len(top),
        "n_prioritized": n_prioritized,
        "rank_col": EXPECTED_PRIMARY_RANK,
    })

    print(f"[DONE] {cond}: rows={n_rows:,} prioritized={n_prioritized:,}")

    # Release the largest objects before moving to the next condition.
    del adata_cond, res, res_sorted, prioritized, top
    del cond_meta, requested_ids, positions, missing_ids, row_positions
    del selected_ids, selected_labels, label_map
    gc.collect()

memory_preflight = pd.DataFrame(memory_records)
expression_audit = pd.DataFrame(expression_records)
historical_result_audit = pd.DataFrame(historical_result_records)
run_summary = pd.DataFrame(run_records)

write_tsv_replace(memory_preflight, METADATA_DIR / f"{OUT_PREFIX}_memory_preflight.tsv")
write_tsv_replace(expression_audit, METADATA_DIR / f"{OUT_PREFIX}_expression_audit.tsv")
write_tsv_replace(historical_result_audit, METADATA_DIR / f"{OUT_PREFIX}_historical_result_checkpoint_audit.tsv")
write_tsv_replace(run_summary, METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv")

display(run_summary)
display(historical_result_audit)

if not historical_result_audit["match"].all():
    raise AssertionError(
        "Condition-specific LIANA result counts drift from the historical execution. "
        "Do not proceed to downstream summaries."
    )



[CONDITION] GBM
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_GBM_celltype_counts_used.tsv (8, 4)


Using resource `consensus`.
Using `.X`!


[LIANA minimal object] AnnData object with n_obs × n_vars = 93563 × 15176
    obs: 'liana_celltype_condition_main'
    uns: 'log1p'


0.29 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 93563 samples and 1210 features
Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


...

Generating ligand-receptor stats for 33099 samples and 1209 features
Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [01:35<00:00, 10.44it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR
[DONE] rGBM: rows=30,423 prioritized=1,676
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_memory_preflight.tsv (4, 9)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_expression_audit.tsv (4, 8)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_historical_result_checkpoint_audit.tsv (8, 5)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_run_summary.tsv (4, 11)


,condition,status,reason,n_cells,n_genes,n_celltypes,included_celltypes,n_liana_rows,n_top_exported,n_prioritized,rank_col
0,GBM,completed,,93563,15176,8,Malignant_AC_like_reactive_malignant;Malignant...,11094,500,662,magnitude_rank
1,LGG,completed,,6333,15176,6,Malignant_AC_like_reactive_malignant;Malignant...,11050,500,625,magnitude_rank
2,ndGBM,completed,,49830,15176,8,Malignant_AC_like_reactive_malignant;Malignant...,23603,500,1337,magnitude_rank
3,rGBM,completed,,33099,15176,8,Malignant_AC_like_reactive_malignant;Malignant...,30423,500,1676,magnitude_rank


,condition,checkpoint,expected_historical,observed,match
0,GBM,n_liana_rows,11094,11094,True
1,GBM,n_prioritized,662,662,True
2,LGG,n_liana_rows,11050,11050,True
3,LGG,n_prioritized,625,625,True
4,ndGBM,n_liana_rows,23603,23603,True
5,ndGBM,n_prioritized,1337,1337,True
6,rGBM,n_liana_rows,30423,30423,True
7,rGBM,n_prioritized,1676,1676,True


In [10]:
# Combine and save canonical condition-specific LIANA results
if len(all_results) != len(CONDITIONS_ORDER):
    raise RuntimeError("Not all four condition-specific LIANA runs completed.")

all_res = pd.concat(all_results, ignore_index=True)
top_res = pd.concat(top_results, ignore_index=True)
prioritized_res = pd.concat(prioritized_results, ignore_index=True)

aggregate_checkpoints = pd.DataFrame([
    {
        "checkpoint": "total_liana_rows",
        "expected_historical": EXPECTED_TOTAL_LIANA_ROWS,
        "observed": len(all_res),
        "match": len(all_res) == EXPECTED_TOTAL_LIANA_ROWS,
    },
    {
        "checkpoint": "total_prioritized_rows",
        "expected_historical": EXPECTED_TOTAL_PRIORITIZED,
        "observed": len(prioritized_res),
        "match": len(prioritized_res) == EXPECTED_TOTAL_PRIORITIZED,
    },
    {
        "checkpoint": "top_export_rows",
        "expected_historical": 2_000,
        "observed": len(top_res),
        "match": len(top_res) == 2_000,
    },
])
write_tsv_replace(
    aggregate_checkpoints,
    METADATA_DIR / f"{OUT_PREFIX}_aggregate_result_checkpoint_audit.tsv",
)
display(aggregate_checkpoints)

if not aggregate_checkpoints["match"].all():
    raise AssertionError("Aggregate historical LIANA checkpoints do not match.")

all_path = METADATA_DIR / f"{OUT_PREFIX}_all_results.tsv.gz"
if all_path.exists():
    all_path.unlink()
all_res.to_csv(all_path, sep="\t", index=False, compression="gzip")
print("[WRITE]", all_path, all_res.shape)

write_tsv_replace(
    top_res,
    METADATA_DIR / f"{OUT_PREFIX}_top_interactions_per_condition.tsv",
)
write_tsv_replace(
    prioritized_res,
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_interactions.tsv",
)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_aggregate_result_checkpoint_audit.tsv (3, 4)


,checkpoint,expected_historical,observed,match
0,total_liana_rows,76170,76170,True
1,total_prioritized_rows,4300,4300,True
2,top_export_rows,2000,2000,True


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_all_results.tsv.gz (76170, 26)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_top_interactions_per_condition.tsv (2000, 26)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_prioritized_interactions.tsv (4300, 27)


WindowsPath('G:/Repository/glioma-cnv-single-cell-atlas/metadata/LIANA_condition_specific_HVG8000_r0_6_prioritized_interactions.tsv')

In [11]:
# Canonical summaries
source_target_summary = summarize_source_target(prioritized_res)
axis_summary = summarize_axis(prioritized_res, "axis")
biological_axis_summary = summarize_axis(prioritized_res, "biological_axis")
lr_summary = summarize_lr(prioritized_res)

write_tsv_replace(
    source_target_summary,
    METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv",
)
write_tsv_replace(
    axis_summary,
    METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv",
)
write_tsv_replace(
    biological_axis_summary,
    METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv",
)
write_tsv_replace(
    lr_summary,
    METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv",
)

expected_axes = {
    "malignant_or_cycling_to_malignant_or_cycling",
    "malignant_or_cycling_to_microenvironment",
    "microenvironment_to_malignant_or_cycling",
    "microenvironment_to_microenvironment",
}
observed_axes = set(axis_summary["axis"].astype(str))
if observed_axes != expected_axes:
    raise AssertionError(
        f"Canonical condition-specific axis set mismatch. "
        f"Missing={sorted(expected_axes-observed_axes)} "
        f"Unexpected={sorted(observed_axes-expected_axes)}"
    )

display(axis_summary)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_source_target_summary.tsv (228, 11)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_axis_summary.tsv (16, 7)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_biological_axis_summary.tsv (228, 7)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_ligand_receptor_summary.tsv (912, 9)


,condition,axis,n_interactions,n_unique_source_target,n_unique_lr,min_rank,median_rank
0,GBM,malignant_or_cycling_to_malignant_or_cycling,292,25,64,1.169720e-06,0.016145
2,GBM,microenvironment_to_malignant_or_cycling,155,15,41,6.580068e-07,0.017523
3,GBM,microenvironment_to_microenvironment,108,9,60,7.312065e-08,0.014730
1,GBM,malignant_or_cycling_to_microenvironment,107,15,34,6.139099e-05,0.018508
4,LGG,malignant_or_cycling_to_malignant_or_cycling,292,16,66,2.947987e-07,0.010211
5,LGG,malignant_or_cycling_to_microenvironment,138,8,58,2.386556e-04,0.019051
6,LGG,microenvironment_to_malignant_or_cycling,122,8,40,2.652548e-06,0.013154
7,LGG,microenvironment_to_microenvironment,73,4,51,5.174519e-04,0.014267
8,ndGBM,malignant_or_cycling_to_malignant_or_cycling,427,25,75,1.615459e-08,0.011631
10,ndGBM,microenvironment_to_malignant_or_cycling,336,15,101,1.033689e-06,0.017394


In [12]:
# Cross-condition prioritized interaction presence
presence = (
    prioritized_res.groupby(
        [
            "interaction_key", "source_clean", "target_clean", "lr_pair",
            "ligand", "receptor", "axis", "biological_axis",
        ],
        observed=True,
    )
    .agg(
        n_conditions=("condition", "nunique"),
        conditions=(
            "condition",
            lambda x: ";".join(sorted(pd.Series(x).astype(str).unique())),
        ),
        min_rank_all=("rank_for_sorting", "min"),
        median_rank_all=("rank_for_sorting", "median"),
    )
    .reset_index()
    .sort_values(["n_conditions", "min_rank_all"], ascending=[False, True])
)

rank_pivot = prioritized_res.pivot_table(
    index="interaction_key",
    columns="condition",
    values="rank_for_sorting",
    aggfunc="min",
).reset_index()
rank_pivot.columns = [
    str(c) if c == "interaction_key" else f"rank_{c}"
    for c in rank_pivot.columns
]
presence = presence.merge(rank_pivot, on="interaction_key", how="left")

write_tsv_replace(
    presence,
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_interaction_condition_presence.tsv",
)

presence_audit = pd.DataFrame([{
    "checkpoint": "presence_rows",
    "expected_historical": EXPECTED_PRESENCE_ROWS,
    "observed": len(presence),
    "match": len(presence) == EXPECTED_PRESENCE_ROWS,
}])
write_tsv_replace(
    presence_audit,
    METADATA_DIR / f"{OUT_PREFIX}_presence_checkpoint_audit.tsv",
)
display(presence_audit)

if not presence_audit["match"].all():
    raise AssertionError("Cross-condition presence table row count drifted from historical execution.")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_prioritized_interaction_condition_presence.tsv (2450, 16)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_presence_checkpoint_audit.tsv (1, 4)


,checkpoint,expected_historical,observed,match
0,presence_rows,2450,2450,True


In [13]:
# Focused malignant/microenvironment outputs
focused_axes = [
    "malignant_or_cycling_to_microenvironment",
    "microenvironment_to_malignant_or_cycling",
    "malignant_or_cycling_to_malignant_or_cycling",
]
focused = prioritized_res.loc[prioritized_res["axis"].isin(focused_axes)].copy()
focused = focused.sort_values(
    ["condition", "axis", "rank_for_sorting"],
    ascending=[True, True, True],
)

keywords = [
    "PTN", "MDK", "PTPRZ1", "NCL", "SDC3",
    "TIMP1", "CD63",
    "LGALS1", "ITGB1",
    "MIF", "EGFR", "CD74",
    "VIM", "CD44",
    "VEGFA", "KDR", "FLT1", "ITGB1",
    "APP", "APOE", "LRP1",
    "TGFB", "COL", "FN1", "SPP1", "CXCL", "CCL",
]
pattern = "|".join(keywords)
focused["matches_candidate_keyword"] = focused["lr_pair"].str.contains(
    pattern, case=False, regex=True
)

candidate_interactions = focused.loc[focused["matches_candidate_keyword"]].copy()

candidate_summary = (
    candidate_interactions.groupby(["condition", "axis", "lr_pair"], observed=True)
    .agg(
        n_source_target=("source_target", "nunique"),
        min_rank=("rank_for_sorting", "min"),
        top_source_target=(
            "source_target",
            lambda x: ";".join(pd.Series(x).astype(str).head(5)),
        ),
    )
    .reset_index()
    .sort_values(
        ["condition", "n_source_target", "min_rank"],
        ascending=[True, False, True],
    )
)

write_tsv_replace(
    focused,
    METADATA_DIR / f"{OUT_PREFIX}_focused_malignant_microenvironment_interactions.tsv",
)
write_tsv_replace(
    candidate_interactions,
    METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_interactions_annotated.tsv",
)
write_tsv_replace(
    candidate_summary,
    METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_interaction_summary.tsv",
)

focused_audit = pd.DataFrame([
    {
        "checkpoint": "focused_rows",
        "expected_historical": EXPECTED_FOCUSED_ROWS,
        "observed": len(focused),
        "match": len(focused) == EXPECTED_FOCUSED_ROWS,
    },
    {
        "checkpoint": "candidate_rows",
        "expected_historical": EXPECTED_CANDIDATE_ROWS,
        "observed": len(candidate_interactions),
        "match": len(candidate_interactions) == EXPECTED_CANDIDATE_ROWS,
    },
    {
        "checkpoint": "candidate_summary_rows",
        "expected_historical": EXPECTED_CANDIDATE_SUMMARY_ROWS,
        "observed": len(candidate_summary),
        "match": len(candidate_summary) == EXPECTED_CANDIDATE_SUMMARY_ROWS,
    },
])
write_tsv_replace(
    focused_audit,
    METADATA_DIR / f"{OUT_PREFIX}_focused_output_checkpoint_audit.tsv",
)
display(focused_audit)

if not focused_audit["match"].all():
    raise AssertionError("Focused/candidate output counts drifted from historical execution.")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_focused_malignant_microenvironment_interactions.tsv (3465, 28)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_candidate_axis_interactions_annotated.tsv (2302, 28)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_candidate_axis_interaction_summary.tsv (475, 6)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_focused_output_checkpoint_audit.tsv (3, 4)


,checkpoint,expected_historical,observed,match
0,focused_rows,3465,3465,True
1,candidate_rows,2302,2302,True
2,candidate_summary_rows,475,475,True


In [14]:
# Diagnostic figures — generated for historical continuity; do not stage during Main Analysis phase
figure_records = []

def record_figure(path, description):
    figure_records.append({
        "figure": Path(path).name,
        "path": str(Path(path)),
        "description": description,
    })

if len(axis_summary) > 0:
    mat = axis_summary.pivot_table(
        index="axis",
        columns="condition",
        values="n_interactions",
        fill_value=0,
    )
    mat = mat.reindex(columns=[c for c in CONDITIONS_ORDER if c in mat.columns])
    plt.figure(figsize=(8, 4.5))
    plt.imshow(mat.values, aspect="auto")
    plt.yticks(range(mat.shape[0]), mat.index)
    plt.xticks(range(mat.shape[1]), mat.columns, rotation=45, ha="right")
    plt.colorbar(label="n prioritized LR rows")
    plt.title("LIANA condition-specific axis summary")
    p = savefig(FIGURE_DIR / f"{OUT_PREFIX}_axis_summary_heatmap.png")
    record_figure(p, "Diagnostic condition-by-axis prioritized LR row counts.")

if len(biological_axis_summary) > 0:
    top_bio_axes = (
        biological_axis_summary.groupby("biological_axis", observed=True)["n_interactions"]
        .sum()
        .sort_values(ascending=False)
        .head(20)
        .index
    )
    mat2 = biological_axis_summary.loc[
        biological_axis_summary["biological_axis"].isin(top_bio_axes)
    ].pivot_table(
        index="biological_axis",
        columns="condition",
        values="n_interactions",
        fill_value=0,
    )
    mat2 = mat2.reindex(index=top_bio_axes)
    mat2 = mat2.reindex(columns=[c for c in CONDITIONS_ORDER if c in mat2.columns])
    plt.figure(figsize=(9, 7))
    plt.imshow(mat2.values, aspect="auto")
    plt.yticks(range(mat2.shape[0]), mat2.index)
    plt.xticks(range(mat2.shape[1]), mat2.columns, rotation=45, ha="right")
    plt.colorbar(label="n prioritized LR rows")
    plt.title("Top condition-specific biological LIANA axes")
    p = savefig(FIGURE_DIR / f"{OUT_PREFIX}_biological_axis_summary_heatmap.png")
    record_figure(p, "Diagnostic top biological-axis counts by condition.")

if len(candidate_summary) > 0:
    top_lrs = (
        candidate_summary.groupby("lr_pair", observed=True)["n_source_target"]
        .sum()
        .sort_values(ascending=False)
        .head(20)
        .index
    )
    mat3 = candidate_summary.loc[
        candidate_summary["lr_pair"].isin(top_lrs)
    ].pivot_table(
        index="lr_pair",
        columns="condition",
        values="n_source_target",
        fill_value=0,
    )
    mat3 = mat3.reindex(index=top_lrs)
    mat3 = mat3.reindex(columns=[c for c in CONDITIONS_ORDER if c in mat3.columns])
    plt.figure(figsize=(8, 7))
    plt.imshow(mat3.values, aspect="auto")
    plt.yticks(range(mat3.shape[0]), mat3.index)
    plt.xticks(range(mat3.shape[1]), mat3.columns, rotation=45, ha="right")
    plt.colorbar(label="n source-target pairs")
    plt.title("Candidate LR axes across conditions")
    p = savefig(FIGURE_DIR / f"{OUT_PREFIX}_candidate_LR_condition_heatmap.png")
    record_figure(p, "Diagnostic candidate LR source-target counts by condition.")

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(
    figure_manifest,
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
)
display(figure_manifest)


[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_condition_specific_HVG8000_r0_6\LIANA_condition_specific_HVG8000_r0_6_axis_summary_heatmap.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_condition_specific_HVG8000_r0_6\LIANA_condition_specific_HVG8000_r0_6_biological_axis_summary_heatmap.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_condition_specific_HVG8000_r0_6\LIANA_condition_specific_HVG8000_r0_6_candidate_LR_condition_heatmap.png
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_figure_manifest.tsv (3, 3)


,figure,path,description
0,LIANA_condition_specific_HVG8000_r0_6_axis_sum...,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Diagnostic condition-by-axis prioritized LR ro...
1,LIANA_condition_specific_HVG8000_r0_6_biologic...,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Diagnostic top biological-axis counts by condi...
2,LIANA_condition_specific_HVG8000_r0_6_candidat...,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Diagnostic candidate LR source-target counts b...


In [15]:
# Final audit summary and output manifest
summary = pd.DataFrame([
    {"item": "post_upstream_label_canonicalization", "value": True},
    {"item": "historical_condition_membership_preserved", "value": bool(run_plan_audit["match"].all())},
    {"item": "historical_result_checkpoints_all_match", "value": bool(historical_result_audit["match"].all())},
    {"item": "aggregate_result_checkpoints_all_match", "value": bool(aggregate_checkpoints["match"].all())},
    {"item": "presence_checkpoint_match", "value": bool(presence_audit["match"].all())},
    {"item": "focused_output_checkpoints_all_match", "value": bool(focused_audit["match"].all())},
    {"item": "n_conditions_completed", "value": int((run_summary["status"] == "completed").sum())},
    {"item": "total_liana_rows", "value": len(all_res)},
    {"item": "total_prioritized_rows", "value": len(prioritized_res)},
    {"item": "primary_rank_col", "value": EXPECTED_PRIMARY_RANK},
    {"item": "rank_threshold", "value": RANK_THRESHOLD},
    {"item": "liana_version", "value": LIANA_VERSION},
    {"item": "n_perms", "value": N_PERMS},
    {"item": "seed", "value": SEED},
    {"item": "n_jobs", "value": N_JOBS},
    {"item": "memory_safe_streamed_counts", "value": True},
    {"item": "historical_scanpy_preprocessing_reproduced", "value": True},
    {"item": "group_label_mode", "value": "canonical_notebook25_direct"},
    {"item": "historical_aliases_used_for_computation", "value": False},
    {"item": "runtime_versions_recorded_in_run_config", "value": True},
    {"item": "historical_bitwise_score_equivalence_asserted", "value": False},
    {"item": "completed_at", "value": datetime.now().isoformat(timespec="seconds")},
])
write_tsv_replace(
    summary,
    METADATA_DIR / f"{OUT_PREFIX}_canonical_run_summary.tsv",
)
display(summary)

manifest_paths = [
    METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_resolved_columns.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_all.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_condition_celltype_inclusion_decisions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_condition_run_plan.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_historical_membership_checkpoint_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_historical_to_canonical_label_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_input_membership_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_run_config.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_memory_preflight.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_expression_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_historical_result_checkpoint_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_aggregate_result_checkpoint_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_all_results.tsv.gz",
    METADATA_DIR / f"{OUT_PREFIX}_top_interactions_per_condition.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_interaction_condition_presence.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_presence_checkpoint_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_focused_malignant_microenvironment_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_interactions_annotated.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_interaction_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_focused_output_checkpoint_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_canonical_run_summary.tsv",
]
for cond in CONDITIONS_ORDER:
    manifest_paths.append(
        METADATA_DIR / f"{OUT_PREFIX}_{cond}_celltype_counts_used.tsv"
    )

manifest = pd.DataFrame([
    {
        "file": p.name,
        "path": str(p),
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / (1024 ** 2), 3) if p.exists() else np.nan,
    }
    for p in manifest_paths
])
write_tsv_replace(
    manifest,
    METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv",
)
display(manifest)

if not manifest["exists"].all():
    raise AssertionError("One or more expected Notebook 28 outputs are missing.")

# Final hard gate.
final_flags = [
    bool(run_plan_audit["match"].all()),
    bool(historical_result_audit["match"].all()),
    bool(aggregate_checkpoints["match"].all()),
    bool(presence_audit["match"].all()),
    bool(focused_audit["match"].all()),
]
if not all(final_flags):
    raise AssertionError("Notebook 28 final validation gate failed.")

gc.collect()
print("[DONE] Notebook 28 completed successfully.")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_canonical_run_summary.tsv (22, 2)


,item,value
0,post_upstream_label_canonicalization,True
1,historical_condition_membership_preserved,True
2,historical_result_checkpoints_all_match,True
3,aggregate_result_checkpoints_all_match,True
4,presence_checkpoint_match,True
5,focused_output_checkpoints_all_match,True
6,n_conditions_completed,4
7,total_liana_rows,76170
8,total_prioritized_rows,4300
9,primary_rank_col,magnitude_rank


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_condition_specific_HVG8000_r0_6_output_manifest.tsv (33, 4)


,file,path,exists,size_mb
0,LIANA_condition_specific_HVG8000_r0_6_readines...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
1,LIANA_condition_specific_HVG8000_r0_6_resolved...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
2,LIANA_condition_specific_HVG8000_r0_6_celltype...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
3,LIANA_condition_specific_HVG8000_r0_6_conditio...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.004
4,LIANA_condition_specific_HVG8000_r0_6_conditio...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
5,LIANA_condition_specific_HVG8000_r0_6_historic...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
6,LIANA_condition_specific_HVG8000_r0_6_historic...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
7,LIANA_condition_specific_HVG8000_r0_6_input_me...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
8,LIANA_condition_specific_HVG8000_r0_6_run_conf...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
9,LIANA_condition_specific_HVG8000_r0_6_memory_p...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000


[DONE] Notebook 28 completed successfully.
